# ARX Algorithm-Only Notebook

Notebook nay chi chay phan thuat toan ARX:
1. Nap/sinh du lieu
2. Chia train/validation/test theo thu tu thoi gian
3. Tao regression matrix
4. Uoc luong OLS
5. Danh gia 1-step, 12-step, free-run
6. Residual diagnostics
7. Model structure search
8. Luu artifact JSON

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from arx_pipeline import (
    DataConfig,
    SplitConfig,
    ModelConfig,
    load_or_generate_data,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    build_true_theta,
    summarize_parameters,
    compute_ar_roots,
    evaluate_slice,
    model_selection_search,
    evaluate_candidate_order,
    artifact_payload,
)

DATA_CONFIG = DataConfig(
    csv_path=Path("greenhouse_data.csv"),
    generator_script_path=Path("data_generator.py"),
    force_regenerate_from_script=False,
    auto_save_generated_csv=True,
    generated_days=365,
    generated_sampling_seconds=300,
    generated_seed=42,
    generated_start_date="2025-01-01",
)

SPLIT_CONFIG = SplitConfig(train_ratio=0.60, val_ratio=0.20)
MODEL_CONFIG = ModelConfig(na=2, nb=2, nk=1, include_intercept=False)

In [2]:
df_full, true_params, data_source = load_or_generate_data(DATA_CONFIG)
df_train, df_val, df_test = split_time_series(df_full, SPLIT_CONFIG)

overview = pd.Series({
    "data_source": data_source,
    "rows_full": len(df_full),
    "rows_train": len(df_train),
    "rows_val": len(df_val),
    "rows_test": len(df_test),
    "timestamp_start": str(df_full["Timestamp"].iloc[0]),
    "timestamp_end": str(df_full["Timestamp"].iloc[-1]),
    "months_present": sorted(int(m) for m in pd.Series(df_full["Month"]).dropna().unique()),
    "seasons_present": sorted(str(s) for s in pd.Series(df_full["Season"]).dropna().unique()),
})

overview

data_source                        CSV:greenhouse_data.csv
rows_full                                           105120
rows_train                                           63072
rows_val                                             21024
rows_test                                            21024
timestamp_start                        2025-01-01 00:00:00
timestamp_end                          2025-12-31 23:55:00
months_present     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
seasons_present           [autumn, spring, summer, winter]
dtype: object

In [3]:
x_train, y_train = build_regression_matrix(df_train, MODEL_CONFIG)
theta_hat, cov_hat, sigma2_hat = estimate_ols(x_train, y_train)
true_theta = build_true_theta(true_params, MODEL_CONFIG)

train_matrix_info = pd.Series({
    "x_train_shape": x_train.shape,
    "y_train_shape": y_train.shape,
    "rank_x_train": int(np.linalg.matrix_rank(x_train)),
    "n_params": len(MODEL_CONFIG.param_names),
    "cond_xtx": float(np.linalg.cond(x_train.T @ x_train)),
    "sigma2_hat": float(sigma2_hat),
})

train_matrix_info

x_train_shape        (63070, 14)
y_train_shape           (63070,)
rank_x_train                  14
n_params                      14
cond_xtx         56303331.052535
sigma2_hat               0.06248
dtype: object

In [4]:
params_df = pd.DataFrame(
    summarize_parameters(theta_hat, cov_hat, MODEL_CONFIG, true_params)
)
roots_df = pd.DataFrame(compute_ar_roots(theta_hat, MODEL_CONFIG))

params_display = params_df[[
    "name",
    "estimate",
    "std",
    "ci95_low",
    "ci95_high",
    "true_value",
    "delta_vs_true",
    "sign_ok",
]].copy()

sign_ok_count = int(params_df["sign_ok"].sum())
sign_total = int(len(params_df))
print(f"Sign recovery: {sign_ok_count}/{sign_total}")

params_display.round(6), roots_df.round(6)

Sign recovery: 14/14


(               name  estimate       std  ci95_low  ci95_high  true_value  \
 0                a1  0.963098  0.001670  0.959824   0.966371     0.96500   
 1                a2  0.026738  0.001724  0.023358   0.030118     0.02500   
 2   b_Temperature_1 -0.007410  0.002548 -0.012403  -0.002416    -0.00800   
 3   b_Temperature_2 -0.003922  0.002493 -0.008808   0.000964    -0.00400   
 4      b_Humidity_1  0.002567  0.000836  0.000929   0.004206     0.00250   
 5      b_Humidity_2  0.001029  0.000825 -0.000588   0.002646     0.00120   
 6         b_Light_1 -0.000132  0.000070 -0.000270   0.000005    -0.00022   
 7         b_Light_2 -0.000194  0.000071 -0.000334  -0.000055    -0.00010   
 8          b_Drip_1  1.251167  0.003388  1.244526   1.257809     1.25000   
 9          b_Drip_2  1.851585  0.005452  1.840899   1.862271     1.85000   
 10         b_Mist_1  0.038049  0.010848  0.016787   0.059311     0.05000   
 11         b_Mist_2  0.035212  0.005013  0.025385   0.045038     0.03000   

In [5]:
train_eval = evaluate_slice("Train", df_train, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)
val_eval = evaluate_slice("Validation", df_val, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)
test_eval = evaluate_slice("Test", df_test, theta_hat, MODEL_CONFIG, true_theta=true_theta, n_step=12)

metrics_table = pd.DataFrame([
    {
        "Split": "Train",
        "FIT_1step": train_eval["metrics_1step"]["FIT"],
        "FIT_12step": train_eval["metrics_n_step"]["FIT"],
        "FIT_sim": train_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": train_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": train_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": train_eval["metrics_sim"]["RMSE"],
    },
    {
        "Split": "Validation",
        "FIT_1step": val_eval["metrics_1step"]["FIT"],
        "FIT_12step": val_eval["metrics_n_step"]["FIT"],
        "FIT_sim": val_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": val_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": val_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": val_eval["metrics_sim"]["RMSE"],
    },
    {
        "Split": "Test",
        "FIT_1step": test_eval["metrics_1step"]["FIT"],
        "FIT_12step": test_eval["metrics_n_step"]["FIT"],
        "FIT_sim": test_eval["metrics_sim"]["FIT"],
        "Theo_FIT_sim": test_eval.get("theoretical_max_free_run", {}).get("FIT", np.nan),
        "RMSE_1step": test_eval["metrics_1step"]["RMSE"],
        "RMSE_sim": test_eval["metrics_sim"]["RMSE"],
    },
])

metrics_table.round(4)

,Split,FIT_1step,FIT_12step,FIT_sim,Theo_FIT_sim,RMSE_1step,RMSE_sim
0,Train,92.5471,76.4447,49.5463,49.0595,0.2499,1.6920
1,Validation,91.6406,73.1094,42.9588,42.2497,0.2510,1.7124
2,Test,91.3491,72.6736,43.8749,43.2691,0.2520,1.6347


In [6]:
diag_table = pd.DataFrame([
    {
        "Split": "Validation",
        "mean": val_eval["residual_diagnostics"]["mean"],
        "std": val_eval["residual_diagnostics"]["std"],
        "shapiro_p": val_eval["residual_diagnostics"]["normality"]["shapiro_pvalue"],
        "dagostino_p": val_eval["residual_diagnostics"]["normality"]["dagostino_pvalue"],
        "ljung_box_pass": val_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"],
        "failed_lags": val_eval["residual_diagnostics"]["ljung_box"]["failed_lags"],
    },
    {
        "Split": "Test",
        "mean": test_eval["residual_diagnostics"]["mean"],
        "std": test_eval["residual_diagnostics"]["std"],
        "shapiro_p": test_eval["residual_diagnostics"]["normality"]["shapiro_pvalue"],
        "dagostino_p": test_eval["residual_diagnostics"]["normality"]["dagostino_pvalue"],
        "ljung_box_pass": test_eval["residual_diagnostics"]["ljung_box"]["passes_all_lags"],
        "failed_lags": test_eval["residual_diagnostics"]["ljung_box"]["failed_lags"],
    },
])

diag_table.round(4)

,Split,mean,std,shapiro_p,dagostino_p,ljung_box_pass,failed_lags
0,Validation,0.0016,0.251,0.2024,0.4118,True,[]
1,Test,0.0005,0.252,0.7259,0.1141,True,[]


In [7]:
selection_df = model_selection_search(
    df_train=df_train,
    df_val=df_val,
    base_model_config=MODEL_CONFIG,
    na_list=[1, 2, 3],
    nb_list=[1, 2, 3],
    nk_list=[1, 2],
)

selection_df.head(10).round(4)

,na,nb,nk,n_params,RMSE_1step,FIT_1step,R2_1step,AIC_1step,BIC_1step,RMSE_sim,FIT_sim,R2_sim
0,3,1,1,9,0.4102,86.3359,0.9813,-37443.8834,-37372.3039,1.5035,49.9185,0.7492
1,2,1,1,8,0.4209,85.9812,0.9803,-36371.2618,-36307.6352,1.6094,46.3905,0.7126
2,1,2,1,13,0.2514,91.6262,0.9930,-58026.2945,-57922.9013,1.6211,46.0003,0.7084
3,3,1,2,9,0.4516,84.9589,0.9774,-33407.2784,-33335.6989,1.7004,43.3621,0.6792
4,1,3,1,19,0.2511,91.6375,0.9930,-58066.9289,-57915.8166,1.7079,43.1125,0.6764
5,3,2,1,15,0.2509,91.6417,0.9930,-58096.4441,-57977.1449,1.7116,42.9892,0.6750
6,2,2,1,14,0.2510,91.6406,0.9930,-58096.5147,-57985.1682,1.7124,42.9588,0.6746
7,2,3,1,20,0.2510,91.6406,0.9930,-58080.5684,-57921.5029,1.7201,42.7066,0.6717
8,3,3,1,21,0.2510,91.6406,0.9930,-58078.5459,-57911.5271,1.7216,42.6546,0.6712
9,2,1,2,8,0.4542,84.8712,0.9771,-33167.4943,-33103.8677,1.7320,42.3082,0.6672


In [8]:
if selection_df.empty:
    raise RuntimeError("Model selection rong, kiem tra lai du lieu/config")

best_row = selection_df.iloc[0]
best_candidate = evaluate_candidate_order(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    base_model_config=MODEL_CONFIG,
    na=int(best_row["na"]),
    nb=int(best_row["nb"]),
    nk=int(best_row["nk"]),
)

best_cfg = best_candidate["model_config"]
comparison_df = pd.DataFrame([
    {
        "Model": f"Baseline ARX({MODEL_CONFIG.na},{MODEL_CONFIG.nb},{MODEL_CONFIG.nk})",
        "Val_FIT_sim": val_eval["metrics_sim"]["FIT"],
        "Test_FIT_sim": test_eval["metrics_sim"]["FIT"],
        "Val_RMSE_sim": val_eval["metrics_sim"]["RMSE"],
        "Test_RMSE_sim": test_eval["metrics_sim"]["RMSE"],
    },
    {
        "Model": f"Best ARX({best_cfg.na},{best_cfg.nb},{best_cfg.nk})",
        "Val_FIT_sim": best_candidate["val"]["metrics_sim"]["FIT"],
        "Test_FIT_sim": best_candidate["test"]["metrics_sim"]["FIT"],
        "Val_RMSE_sim": best_candidate["val"]["metrics_sim"]["RMSE"],
        "Test_RMSE_sim": best_candidate["test"]["metrics_sim"]["RMSE"],
    },
])

comparison_df.round(4)

,Model,Val_FIT_sim,Test_FIT_sim,Val_RMSE_sim,Test_RMSE_sim
0,"Baseline ARX(2,2,1)",42.9588,43.8749,1.7124,1.6347
1,"Best ARX(3,1,1)",49.9185,48.1578,1.5035,1.5100


In [9]:
results_algo = {
    "data_source": data_source,
    "data_config": DATA_CONFIG,
    "split_config": SPLIT_CONFIG,
    "model_config": MODEL_CONFIG,
    "df_full": df_full,
    "df_train": df_train,
    "df_val": df_val,
    "df_test": df_test,
    "true_params": true_params,
    "dataset_overview": {
        "rows": int(len(df_full)),
        "timestamp_start": str(df_full["Timestamp"].iloc[0]),
        "timestamp_end": str(df_full["Timestamp"].iloc[-1]),
        "months_present": sorted(int(m) for m in pd.Series(df_full["Month"]).dropna().unique()),
        "seasons_present": sorted(str(s) for s in pd.Series(df_full["Season"]).dropna().unique()),
        "condition_number_xtx": float(np.linalg.cond(x_train.T @ x_train)),
        "rank_x_train": int(np.linalg.matrix_rank(x_train)),
        "n_params": int(len(MODEL_CONFIG.param_names)),
    },
    "theta_hat": theta_hat.tolist(),
    "sigma2": float(sigma2_hat),
    "ar_roots": roots_df.to_dict(orient="records"),
    "parameter_summary": params_df.to_dict(orient="records"),
    "train": train_eval,
    "val": val_eval,
    "test": test_eval,
    "model_selection": selection_df,
    "best_candidate": best_candidate,
}

payload = artifact_payload(results_algo)
output_path = Path("arx_model_algo_only.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
    f.write("\n")

print(f"Saved: {output_path}")

Saved: arx_model_algo_only.json
